# Absolute colour–magnitude diagram of Gaia DR4 microlensing sources

Use notebook 02's joined CSV to plot Paczyński-0 baseline source colours and
absolute G magnitudes derived from Paczyński-3 source parallax. Apply explicit
fit-level and catalogue-quality cuts, with optional extinction corrections.
Save PNG/PDF figures and the selected-source CSV.

An optional Gaia DR3 reference sample can provide a density background. The
default makes a source-only diagram, so no external reference file is required.


The data are described in the Gaia DR4 publication by **Wyrzykowski et al. (2026)**.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from astropy.table import Table
from matplotlib.colors import LogNorm


def find_project_root(start):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "01_archive_access.ipynb").is_file():
            return candidate
    raise FileNotFoundError("Open this notebook from the gaia-dr4-microlensing-notebooks folder.")

ROOT = find_project_root(Path.cwd())

INPUT_PATH = ROOT / 'outputs' / 'gaiadr4' / 'gaia_dr4_microlensing_events_joined.csv'
REFERENCE_PATH = ROOT / 'data' / 'gaia_dr3_cmd_reference_extinction.vot.gz'
OUTPUT_STEM = ROOT / 'outputs' / 'gaiadr4' / 'microlensing_sources_absolute_cmd_quality_cut'
DEREDDEN = True
USE_REFERENCE_BACKGROUND = False  # See data/README.md for the optional DR3 sample

# Catalogue cuts: conservative Gaia/all_source photometric and astrometric reliability.
MAX_RUWE = 1.4
MIN_VISIBILITY_PERIODS = 9
MIN_G_FLUX_SNR = 50
MIN_BP_RP_FLUX_SNR = 20

if not INPUT_PATH.exists():
    raise FileNotFoundError(f'Missing pre-downloaded joined table: {INPUT_PATH}')
print(INPUT_PATH)
print(f'Dereddening: {DEREDDEN}')

In [ ]:
events = Table.read(INPUT_PATH, format='ascii.csv')
print(f'Loaded {len(events):,} joined microlensing rows with {len(events.colnames):,} columns.')

def numeric(column):
    """Return an astropy column as floats, converting masked values to NaN."""
    return np.asarray(np.ma.asarray(column, dtype=float).filled(np.nan), dtype=float)

def boolean(column):
    """Read Gaia boolean columns while treating an absent/masked value as False."""
    value = np.ma.asarray(column).filled(False)
    if value.dtype.kind in 'b':
        return np.asarray(value, dtype=bool)
    return np.char.lower(np.asarray(value, dtype=str)) == 'true'

# Paczynski-0 baseline source photometry and Paczynski-3 source parallax.
g0 = numeric(events['paczynski0_g0'])
g0_error = numeric(events['paczynski0_g0_error'])
bp0 = numeric(events['paczynski0_bp0'])
bp0_error = numeric(events['paczynski0_bp0_error'])
rp0 = numeric(events['paczynski0_rp0'])
rp0_error = numeric(events['paczynski0_rp0_error'])
source_parallax = numeric(events['paczynski3_varpi_source'])
source_parallax_error = numeric(events['paczynski3_varpi_source_error'])

colour = bp0 - rp0
colour_error = np.hypot(bp0_error, rp0_error)
source_parallax_snr = np.divide(
    source_parallax, source_parallax_error,
    out=np.full(len(events), np.nan), where=source_parallax_error > 0,
)

# Gaia/all_source columns available in the pre-downloaded joined table.
ruwe = numeric(events['ruwe'])
visibility_periods = numeric(events['visibility_periods_used'])
g_flux_snr = numeric(events['phot_g_mean_flux_over_error'])
bp_flux_snr = numeric(events['phot_bp_mean_flux_over_error'])
rp_flux_snr = numeric(events['phot_rp_mean_flux_over_error'])
unreliable_g = boolean(events['unreliable_g_photometry'])

# Fit-level CMD requirements inherited from the current absolute-CMD generation.
fit_quality = (
    np.isfinite(g0) & np.isfinite(g0_error)
    & np.isfinite(bp0) & np.isfinite(rp0)
    & np.isfinite(colour_error) & (colour_error < 1.0)
    & np.isfinite(source_parallax) & (source_parallax > 0)
    & np.isfinite(source_parallax_error) & (source_parallax_error > 0)
    & (source_parallax_snr > 2.0)
)

# New Gaia/all_source data-quality selection.
catalogue_quality = (
    ~unreliable_g
    & np.isfinite(ruwe) & (ruwe < MAX_RUWE)
    & np.isfinite(visibility_periods) & (visibility_periods >= MIN_VISIBILITY_PERIODS)
    & np.isfinite(g_flux_snr) & (g_flux_snr > MIN_G_FLUX_SNR)
    & np.isfinite(bp_flux_snr) & (bp_flux_snr > MIN_BP_RP_FLUX_SNR)
    & np.isfinite(rp_flux_snr) & (rp_flux_snr > MIN_BP_RP_FLUX_SNR)
)

valid = fit_quality & catalogue_quality
print(f'Fit-level CMD selection: {fit_quality.sum():,} / {len(events):,}')
print(f'Gaia/all_source quality selection: {catalogue_quality.sum():,} / {len(events):,}')
print(f'Final quality-cut source CMD: {valid.sum():,} / {len(events):,}')

In [ ]:
absolute_g = np.full(len(events), np.nan)
absolute_g[valid] = g0[valid] + 5 * np.log10(source_parallax[valid]) - 10
absolute_g_error = np.full(len(events), np.nan)
absolute_g_error[valid] = np.hypot(
    g0_error[valid],
    (5 / np.log(10)) * source_parallax_error[valid] / source_parallax[valid],
)
plot_colour = colour.copy()

if DEREDDEN:
    ag = numeric(events['ag'])
    ebpminrp = numeric(events['ebpminrp'])
    extinction_quality = np.isfinite(ag) & np.isfinite(ebpminrp)
    valid &= extinction_quality
    plot_colour -= ebpminrp
    absolute_g[valid] -= ag[valid]
    print(f'After finite extinction corrections: {valid.sum():,} sources')

if USE_REFERENCE_BACKGROUND:
    # The Gaia DR3 reference supplies the density background used by the current CMD.
    reference = Table.read(REFERENCE_PATH)
    reference_colour = numeric(reference['bp_rp'])
    reference_g = numeric(reference['phot_g_mean_mag'])
    reference_parallax = numeric(reference['parallax'])
    reference_valid = np.isfinite(reference_colour) & np.isfinite(reference_g) & (reference_parallax > 0)
    reference_colour = reference_colour[reference_valid]
    reference_absolute_g = reference_g[reference_valid] + 5 * np.log10(reference_parallax[reference_valid]) - 10
    if DEREDDEN:
        reference_colour -= numeric(reference['ebpminrp_gspphot'])[reference_valid]
        reference_absolute_g -= numeric(reference['ag_gspphot'])[reference_valid]

In [ ]:
x_range, y_range = (-0.6, 2.7), (-5.0, 10.5)
fig, ax = plt.subplots(figsize=(6.2, 6.2), constrained_layout=True)
if USE_REFERENCE_BACKGROUND:
    density, x_edges, y_edges = np.histogram2d(
        reference_colour, reference_absolute_g, bins=(260, 300), range=(x_range, y_range)
    )
    ax.pcolormesh(
        x_edges, y_edges, density.T, cmap='Greys', alpha=0.62, shading='auto',
        norm=LogNorm(vmin=1, vmax=max(2, density.max())), rasterized=True,
    )
ax.errorbar(
    plot_colour[valid], absolute_g[valid], xerr=colour_error[valid], yerr=absolute_g_error[valid],
    fmt='none', ecolor='#1769aa', elinewidth=0.8, alpha=0.36, capsize=0, rasterized=True, zorder=2,
)
ax.scatter(
    plot_colour[valid], absolute_g[valid], s=22, color='#1769aa', alpha=0.75,
    edgecolors='white', linewidths=0.3, rasterized=True, zorder=3,
)
ax.invert_yaxis()
ax.set(xlim=x_range, ylim=(y_range[1], y_range[0]))
ax.grid(True, color='0.88', linewidth=0.7)
ax.set_axisbelow(True)
if DEREDDEN:
    ax.set_xlabel(r'Dereddened baseline source colour $(BP_0-RP_0)-E(BP-RP)$ [mag]')
    ax.set_ylabel(r'Extinction-corrected absolute source magnitude $M_{G,0}$ [mag]')
else:
    ax.set_xlabel(r'Baseline source colour $BP_0-RP_0$ [mag]')
    ax.set_ylabel(r'Absolute source magnitude $M_{G,0}$ [mag]')
ax.set_title(f'Quality-cut microlensing sources ({valid.sum():,} events)')

suffix = '_dereddened' if DEREDDEN else ''
figure_png = OUTPUT_STEM.with_name(OUTPUT_STEM.name + suffix + '.png')
figure_pdf = OUTPUT_STEM.with_name(OUTPUT_STEM.name + suffix + '.pdf')
fig.savefig(figure_png, dpi=220)
fig.savefig(figure_pdf)
plt.show()
print(figure_png)
print(figure_pdf)

In [ ]:
output = Table()
output['source_id'] = events['source_id'][valid]
output['paczynski0_g0'] = g0[valid]
output['paczynski0_g0_error'] = g0_error[valid]
output['paczynski0_bp_rp'] = colour[valid]
output['paczynski0_bp_rp_error'] = colour_error[valid]
output['paczynski3_varpi_source_mas'] = source_parallax[valid]
output['paczynski3_varpi_source_snr'] = source_parallax_snr[valid]
output['absolute_g'] = absolute_g[valid]
output['absolute_g_error'] = absolute_g_error[valid]
output['ruwe'] = ruwe[valid]
output['visibility_periods_used'] = visibility_periods[valid]
output['phot_g_mean_flux_over_error'] = g_flux_snr[valid]
output['phot_bp_mean_flux_over_error'] = bp_flux_snr[valid]
output['phot_rp_mean_flux_over_error'] = rp_flux_snr[valid]
if DEREDDEN:
    output['ag'] = ag[valid]
    output['ebpminrp'] = ebpminrp[valid]
    output['dereddened_bp_rp'] = plot_colour[valid]

data_path = OUTPUT_STEM.with_name(OUTPUT_STEM.name + ('_dereddened' if DEREDDEN else '') + '_data.csv')
output.write(data_path, format='ascii.csv', overwrite=True)
print(f'Saved {len(output):,} selected sources to {data_path}')
output[:10]